## Install Dependencies


In [1]:
# Dependencis for the notebook
%pip install spacy pandas torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy
# optuna and optional dependencies
%pip install optuna scikit-learn plotly 
# Qwen optimization (optional) dependencies
%pip install flash-linear-attention causal-conv1d
# Mistral dependencies
%pip install accelerate


/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
import matplotlib
import optuna.visualization
import optuna.importance
import json
from pathlib import Path
from collections import OrderedDict
import modules.llms as llm_parsers
import modules.deepparse_parser as deepparse
import modules.libpostal_client as libpostal_client
import modules.token_classifiers as token_classifiers
from modules.utils import compare_preds, format_time, natural_casing
from typing import NamedTuple
from tqdm.auto import tqdm

from typing import Callable

import abc
import contextlib
import pandas as pd
import time
import pprint
import textwrap
import numpy as np
import modules.train_ner as train_ner

# Total number of addresses in the complete BZK data for calculating estimated time of processing
TOTAL_ADDRESSES = 4_394_539

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

ENTITIES = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "District",
    "State",
    "Region",
    "Country",
    "Other"
]

BZK_ADDRESS_FIELDS = [
    'ApplicantCurrentAddress',
    'VictimBirthPlace',
    'VictimCurrentAddress',     
    'ApplicantBirthPlace',
    'VictimDeathPlace'
]

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## Set hardware accelerator


In [3]:
# List available hardware accelerators for PyTorch
import torch
available_accelerators = []
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
        available_accelerators.append(torch.device(f'cuda:{i}'))
elif torch.accelerator.is_available(): # Support other hardware accelators
    available_accelerators.append(torch.accelerator.current_accelerator())
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')

CUDA - available devices:
  0: NVIDIA A100 80GB PCIe


/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [4]:
if available_accelerators:
    device = available_accelerators[0]
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+cu128, Device: cuda:0


## Load Dataset

Read all splits and concat them

In [5]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_merged : pd.DataFrame = pd.concat(
    [bzkopen_train, bzkopen_val, bzkopen_test], 
    keys=['train', 'val', 'test'], 
    names=['split', 'row']
)
display(bzkopen_merged.sample(5))

field  \
split row                            
test  32       ApplicantBirthPlace   
train 407      ApplicantBirthPlace   
      197  ApplicantCurrentAddress   
      430  ApplicantCurrentAddress   
      113  ApplicantCurrentAddress   

                                          FullAddress UnitNumber HouseNumber  \
split row                                                                      
test  32                                   Kiustendil        NaN         NaN   
train 407                                    Bukarest        NaN         NaN   
      197  Sao Paulo/Brasilien, Rua Santa Efigenia 31        NaN          31   
      430                     Hildesheim Lademühle 56        NaN          56   
      113        Eitorf, Bahnhofstr. 35 Reg.Bez. Köln        NaN          35   

                   StreetName Neighborhood        City District Region State  \
split row                                                                      
test  32                  NaN          NaN  Kiustendil      NaN    NaN   NaN   
train 407                 NaN          NaN    Bukarest      NaN    NaN   NaN   
      197  Rua Santa Efigenia          NaN   Sao Paulo      NaN    NaN   NaN   
      430           Lademühle          NaN  Hildesheim      NaN    NaN   NaN   
      113         Bahnhofstr.          NaN      Eitorf     Köln    NaN   NaN   

             Country PostalCode  
split row                        
test  32         NaN        NaN  
train 407        NaN        NaN  
      197  Brasilien        NaN  
      430        NaN        NaN  
      113        NaN        NaN

## Create folds



In [6]:
N_FOLDS = 10

# Set shuffle seed for reproducibility
SEED=4841762


In [7]:
# Define paths for saving models and predictions
models_path = Path("models") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
models_path.mkdir(parents=True, exist_ok=True)
preds_path = Path("experiments_data") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
preds_path.mkdir(parents=True, exist_ok=True)

In [8]:
class Fold(NamedTuple):
    fold_idx: int
    train_data: pd.DataFrame
    val_data: pd.DataFrame
    models_path: Path
    preds_path: Path

In [9]:

shuffled_bzkopen = bzkopen_merged.sample(frac=1, random_state=SEED)

fold_indices = np.array_split(shuffled_bzkopen.index, N_FOLDS)

folds = []

for fold_idx, val_indices in enumerate(fold_indices):
    train_indices = shuffled_bzkopen.index.difference(val_indices)
    fold_models_path = models_path / f"fold_{fold_idx}"
    fold_models_path.mkdir(parents=True, exist_ok=True)
    fold_preds_path = preds_path / f"fold_{fold_idx}"
    fold_preds_path.mkdir(parents=True, exist_ok=True)
    folds.append(Fold(
        fold_idx, 
        shuffled_bzkopen.loc[train_indices], 
        shuffled_bzkopen.loc[val_indices],
        fold_models_path, 
        fold_preds_path
    ))
display(pd.DataFrame(
    [[len(x) for x in [fold.val_data, fold.train_data]] for fold in folds], 
    columns=['val_fold_size', 'train_fold_size'])
)

,val_fold_size,train_fold_size
0,108,967
1,108,967
2,108,967
3,108,967
4,108,967
5,107,968
6,107,968
7,107,968
8,107,968
9,107,968


# Train Spacy NER models

These models are used for the pattern example matching strategy so they will be reused over multiple different experiments. Therefore it is useful to train them in advance.

In [10]:
for fold in tqdm(folds, unit="fold"):
    output_dir = fold.models_path / 'ner_bzk'
    if output_dir.exists():
        print(f"Fold {fold.fold_idx} already has a trained model, skipping training.")
        continue
    train_ner.train(
        n_iter=30,
        output_dir=output_dir,
        train_df=fold.train_data,
        val_df=fold.val_data
    )

  0%|          | 0/10 [00:00<?, ?fold/s]

Fold 0 already has a trained model, skipping training.
Fold 1 already has a trained model, skipping training.
Fold 2 already has a trained model, skipping training.
Fold 3 already has a trained model, skipping training.
Fold 4 already has a trained model, skipping training.
Fold 5 already has a trained model, skipping training.
Fold 6 already has a trained model, skipping training.
Fold 7 already has a trained model, skipping training.
Fold 8 already has a trained model, skipping training.
Fold 9 already has a trained model, skipping training.


In [ ]:
# Metrics on the required entities (Country, City, StreetName, HouseNumber)
required_results = OrderedDict()
specific_results = OrderedDict()


class FoldEvaluator:
    def __init__(self, model_name : str, supported_entities : list[str]):
        # Metrics on the required entities (Country, City, StreetName, HouseNumber)
        self._required_metric_records = None
        self._specific_metric_records = None
        self.required_metrics = None
        self.specific_metrics = None
        self.model_name = model_name
        self.supported_entities = supported_entities
    
    def __enter__(self):
        self.required_metrics = None
        self.specific_metrics = None
        self._required_metric_records = []
        self._specific_metric_records = []
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        global required_results, specific_results
        self.required_metrics = pd.DataFrame(self._required_metric_records).mean()
        self.specific_metrics = pd.DataFrame(self._specific_metric_records).mean()
        required_results[self.model_name] = self.required_metrics
        specific_results[self.model_name] = self.specific_metrics
        print(f"{self.model_name} - Metrics for Country City Street House:")
        display(self.required_metrics)
        print(f"{self.model_name} - Specific Metrics:")
        display(self.specific_metrics.unstack(level=0))
        self._required_metric_records = None
        self._specific_metric_records = None
    
    def run_on_fold(self, fold : Fold, model):
        assert self._required_metric_records is not None and self._specific_metric_records is not None, "FoldMetricRecorder must be used as a context manager"
        preds_file = fold.preds_path / self.model_name / "preds.json"
        preds_file.parent.mkdir(parents=True, exist_ok=True)
        if preds_file.exists():
            print(f"Fold {fold.fold_idx} already has predictions for {self.model_name}, loading from file.")
            with open(preds_file, "r") as f:
                preds = json.load(f)
        else:
            preds = model.parse_addresses(fold.val_data['FullAddress'].tolist())
            with open(preds_file, "w") as f:
                json.dump(preds, f, indent=4)
        preds_df = pd.DataFrame(preds)
        fold_metrics = compare_preds(preds_df, fold.val_data, target_columns=REQUIRED_ENTITIES)
        self._required_metric_records.append(pd.Series(fold_metrics))
        specific_fold_metrics = OrderedDict()
        for entity in self.supported_entities:
            entity_metrics = compare_preds(preds_df, fold.val_data, target_columns=[entity])
            specific_fold_metrics[entity] = pd.Series(entity_metrics)
        for bzk_field in BZK_ADDRESS_FIELDS:
            mask = fold.val_data['field'] == bzk_field
            field_metrics = compare_preds(preds_df[mask.reset_index(drop=True)], fold.val_data[mask], target_columns=REQUIRED_ENTITIES)
            specific_fold_metrics[bzk_field] = pd.Series(field_metrics)
        self._specific_metric_records.append(pd.concat(specific_fold_metrics, names=['Field/Entity', 'Metric']))
    

In [16]:
with FoldEvaluator("libpostal", libpostal_client.LIBPOSTAL_LABEL_MAPPING.values()) as evaluator:
    model = libpostal_client.LibpostalClient()
    for fold in tqdm(folds, unit="fold"):
        evaluator.run_on_fold(fold, model)
    model.close()

  0%|          | 0/10 [00:00<?, ?fold/s]

libpostal - Metrics for Country City Street House:


accuracy                     0.687913
precision                    0.637836
recall                       0.419072
f1                           0.505555
accuracy_with_tol_1          0.698390
accuracy_with_tol_2          0.706533
accuracy_with_tol_3          0.721655
accuracy_with_tol_4          0.742580
average_levenshtein          2.488696
average_similarity           0.728394
average_levenshtein_match    3.200120
average_similarity_match     0.933376
no_match_rate                0.219784
dtype: float64

libpostal - Specific Metrics:


Field/Entity,HouseNumber,StreetName,City,State,Country,PostalCode,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,,,
accuracy,0.869756,0.771218,0.319064,0.948884,0.791615,0.468354,0.529093,0.809698,0.598449,0.801865,0.820525
precision,0.748952,0.533138,0.615605,0.667500,0.740418,0.033333,0.639412,0.665810,0.621782,0.685237,0.435000
recall,0.679602,0.524948,0.299415,0.468452,0.317047,0.016667,0.411403,0.408604,0.432618,0.433710,0.502778
f1,0.712181,0.528305,0.401641,0.536922,0.441058,0.022222,0.500029,0.501249,0.509248,0.528823,0.455229
accuracy_with_tol_1,0.881862,0.774013,0.336743,0.951670,0.800943,0.468354,0.540611,0.813009,0.619634,0.810099,0.832192
accuracy_with_tol_2,0.906057,0.774013,0.337677,0.952596,0.808385,0.473001,0.553091,0.813009,0.634656,0.810694,0.855774
accuracy_with_tol_3,0.923728,0.780512,0.348849,0.955400,0.833532,0.473001,0.578030,0.830172,0.652770,0.816639,0.866380
accuracy_with_tol_4,0.950727,0.793510,0.374879,0.976791,0.851203,0.475805,0.615146,0.846629,0.664811,0.828181,0.872819
average_levenshtein,0.582321,2.770093,5.511111,0.272300,1.091260,0.501237,3.955788,1.397142,3.355475,1.439254,1.155406


In [17]:
with FoldEvaluator("deepparse", deepparse.DEEPPARSE_LABEL_MAPPING.values()) as evaluator:
    model = deepparse.DeepParseParser(device=device)
    for fold in tqdm(folds, unit="fold"):
        evaluator.run_on_fold(fold, model)
    del model

/home/rpatroni/bzk-post-processing/.venv/lib/python3.12/site-packages/deepparse/download_tools.py:98: UserWarning: The offline parameter is set to False, so if a new pre-trained `bpemb` model is available it will automatically be downloaded.
  warnings.warn(


Loading the embeddings model


  0%|          | 0/10 [00:00<?, ?fold/s]

Vectorizing the address
Vectorizing the address
Vectorizing the address
Vectorizing the address
Vectorizing the address
Vectorizing the address
Vectorizing the address
Vectorizing the address
Vectorizing the address
Vectorizing the address
deepparse - Metrics for Country City Street House:


accuracy                     0.570431
precision                    0.321375
recall                       0.271941
f1                           0.294381
accuracy_with_tol_1          0.577862
accuracy_with_tol_2          0.593216
accuracy_with_tol_3          0.609724
accuracy_with_tol_4          0.630428
average_levenshtein          3.934809
average_similarity           0.648879
average_levenshtein_match    5.110830
average_similarity_match     0.841506
no_match_rate                0.229102
dtype: float64

deepparse - Specific Metrics:


Field/Entity,HouseNumber,City,Country,PostalCode,ApplicantCurrentAddress,VictimBirthPlace,VictimCurrentAddress,ApplicantBirthPlace,VictimDeathPlace
Metric,,,,,,,,,
accuracy,0.864183,0.253037,0.692073,0.922793,0.352289,0.763075,0.443397,0.707778,0.748977
precision,0.834101,0.318456,0.272803,0.025000,0.309011,0.414742,0.313204,0.328656,0.277089
recall,0.656874,0.255519,0.079102,0.016667,0.230186,0.412114,0.265327,0.311501,0.390278
f1,0.733792,0.283273,0.118521,0.020000,0.263626,0.412542,0.287069,0.319081,0.315471
accuracy_with_tol_1,0.886483,0.259536,0.692999,0.928375,0.365328,0.765249,0.458623,0.711914,0.748977
accuracy_with_tol_2,0.939529,0.264183,0.694860,0.939521,0.389361,0.765249,0.506893,0.712555,0.761488
accuracy_with_tol_3,0.963699,0.271625,0.725554,0.948840,0.419085,0.775304,0.535099,0.716106,0.770655
accuracy_with_tol_4,0.976739,0.292108,0.763690,0.956308,0.449611,0.787457,0.560573,0.732547,0.775655
average_levenshtein,0.453141,6.726912,1.825675,0.540247,6.415092,1.823489,5.038260,2.378246,2.282143


In [ ]:
with FoldEvaluator("xlm-roberta-large", token_classifiers.FIGHTING_CRIME_LABEL_MAPPING.values()) as evaluator:
    model = token_classifiers.TokenClassifierAddressParser(
        model_name="hm-haitham/xlm-roberta-large-address-parser", 
        device=device,
        aggregation_strategy="average"
    )
    for fold in tqdm(folds, unit="fold"):
        evaluator.run_on_fold(fold, model)
    del model

In [ ]:
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

prompt0_template = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_0.txt").read_text())

with FoldEvaluator("Qwen3.5-9B-prompt0-0shot", supported_entities) as evaluator:
    model = llm_parsers.QwenAddressParsingModel(
        model_name="Qwen/Qwen3.5-9B", 
        device=device,
        example_strategy=llm_parsers.ZeroShot(),
        prompt=prompt0_template
    )
    for fold in tqdm(folds, unit="fold"):
        evaluator.run_on_fold(fold, model)
    del model

In [ ]:
supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
prompt_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_optuna_best.txt").read_text())
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("Qwen3.5-9B-best-from-optuna", supported_entities) as evaluator:
    model = llm_parsers.QwenAddressParsingModel(
        model_name="Qwen/Qwen3.5-9B", 
        device=device,
        example_strategy=llm_parsers.ZeroShot(),
        prompt=prompt_optuna_best
    )
    for fold in tqdm(folds, unit="fold"):
        pattern_similarity = llm_parsers.NERPatternSimilarExamples(
            example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
            example_labels=fold.train_data,
            labels_to_include=supported_entities,
            num_examples=n_examples,
            model_dir = fold.models_path / "ner_bzk"
        )
        embedding_similarity = llm_parsers.SimilarExamples(
            embedding_model=embedding_model,
            example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
            example_labels=fold.train_data,
            labels_to_include=supported_entities,
            num_examples=n_examples,
            similarity_threshold=similarity_threshold
        )
        hybrid_similarity = llm_parsers.HybridSimilarExamples(
            pattern_similarity=pattern_similarity,
            embedding_similarity=embedding_similarity,
            num_examples=n_examples,
            pool=n_examples
        )
        model.example_strategy = hybrid_similarity
        evaluator.run_on_fold(fold, model)
        model.example_strategy = None # Clear example strategy to save memory
        del hybrid_similarity, embedding_similarity, pattern_similarity
    del model